# Session 3: Types of Addresses: MAC, IP, Subnetting and CIDR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/04-configuring-network-infrastructure/code/session-03-mac-ip-subnetting-cidr.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-03_mac_tool.py`

In [1]:
"""Session 3 - MAC address anatomy: OUI, unicast/multicast and local/universal bits.

Reads a MAC in any common notation (colon, hyphen, Cisco dotted), normalises it,
splits the OUI from the device part, decodes the I/G and U/L bits, and builds
the modified EUI-64 interface ID that IPv6 SLAAC can derive from it.
The small OUI table is a sample for teaching, not a full vendor database.
Run:  python session-03_mac_tool.py
"""
import re

SAMPLE_OUI = {"00:50:56": "VMware", "00:1A:2B": "Example vendor (lab)",
              "3C:5A:B4": "Google", "00:00:0C": "Cisco"}


def normalise(mac: str) -> str:
    hexdigits = re.sub(r"[^0-9a-fA-F]", "", mac)
    if len(hexdigits) != 12:
        raise ValueError(f"{mac!r} is not a 48-bit MAC address")
    return ":".join(hexdigits[i:i + 2] for i in range(0, 12, 2)).upper()


def cisco_format(mac: str) -> str:
    h = normalise(mac).replace(":", "").lower()
    return f"{h[0:4]}.{h[4:8]}.{h[8:12]}"


def describe(mac: str) -> dict:
    m = normalise(mac)
    first = int(m[:2], 16)
    kind = "broadcast" if m == "FF:FF:FF:FF:FF:FF" else ("multicast" if first & 1 else "unicast")
    return {"mac": m, "cisco": cisco_format(m), "oui": m[:8], "nic": m[9:],
            "vendor": SAMPLE_OUI.get(m[:8], "not in sample table"), "type": kind,
            "admin": "locally administered" if first & 2 else "universally administered (burned-in)"}


def eui64(mac: str) -> str:
    """Modified EUI-64: insert FFFE in the middle and flip the U/L bit (bit 7 of byte 1)."""
    b = bytearray(int(x, 16) for x in normalise(mac).split(":"))
    b[0] ^= 0x02
    full = bytes(b[:3]) + b"\xff\xfe" + bytes(b[3:])
    return ":".join(full[i:i + 2].hex() for i in range(0, 8, 2))


if __name__ == "__main__":
    for raw in ["00-50-56-c0-00-08", "0000.0c9f.f00a", "01:00:5E:00:00:05",
                "ff:ff:ff:ff:ff:ff", "3c:5a:b4:12:34:56", "02:42:ac:11:00:02"]:
        d = describe(raw)
        print(f"{raw:20} -> {d['mac']}  ({d['cisco']})")
        print(f"   OUI {d['oui']} [{d['vendor']}], NIC part {d['nic']}, {d['type']}, {d['admin']}")
    print("\nModified EUI-64 interface ID for SLAAC")
    for raw in ["00:1A:2B:3C:4D:5E", "3c:5a:b4:12:34:56"]:
        print(f"  {normalise(raw)} -> {eui64(raw)}  -> fe80::{eui64(raw).lstrip('0')}")

00-50-56-c0-00-08    -> 00:50:56:C0:00:08  (0050.56c0.0008)
   OUI 00:50:56 [VMware], NIC part C0:00:08, unicast, universally administered (burned-in)
0000.0c9f.f00a       -> 00:00:0C:9F:F0:0A  (0000.0c9f.f00a)
   OUI 00:00:0C [Cisco], NIC part 9F:F0:0A, unicast, universally administered (burned-in)
01:00:5E:00:00:05    -> 01:00:5E:00:00:05  (0100.5e00.0005)
   OUI 01:00:5E [not in sample table], NIC part 00:00:05, multicast, universally administered (burned-in)
ff:ff:ff:ff:ff:ff    -> FF:FF:FF:FF:FF:FF  (ffff.ffff.ffff)
   OUI FF:FF:FF [not in sample table], NIC part FF:FF:FF, broadcast, locally administered
3c:5a:b4:12:34:56    -> 3C:5A:B4:12:34:56  (3c5a.b412.3456)
   OUI 3C:5A:B4 [Google], NIC part 12:34:56, unicast, universally administered (burned-in)
02:42:ac:11:00:02    -> 02:42:AC:11:00:02  (0242.ac11.0002)
   OUI 02:42:AC [not in sample table], NIC part 11:00:02, unicast, locally administered

Modified EUI-64 interface ID for SLAAC
  00:1A:2B:3C:4D:5E -> 021a:2bff:fe3c:4d5e  

## `session-03_cidr_planner.py`

In [2]:
"""Session 3 - Subnetting, VLSM and CIDR summarisation with the ipaddress module.

1. Fixed-length subnetting: split a /24 into equal subnets.
2. VLSM: give each department the smallest subnet that fits, largest first.
3. CIDR: summarise contiguous routes into one supernet (route aggregation).
Run:  python session-03_cidr_planner.py
"""
import ipaddress
import math


def fixed_subnets(block, count):
    net = ipaddress.ip_network(block)
    extra_bits = math.ceil(math.log2(count))
    return list(net.subnets(prefixlen_diff=extra_bits))


def prefix_for(hosts):
    """Smallest prefix whose usable hosts (2^h - 2) fit the requirement."""
    h = math.ceil(math.log2(hosts + 2))
    return 32 - h


def vlsm(block, needs):
    """Allocate largest requirement first from the lowest free block that fits."""
    free = [ipaddress.ip_network(block)]
    plan = []
    for name, hosts in sorted(needs, key=lambda x: -x[1]):
        p = prefix_for(hosts)
        candidates = sorted((f for f in free if f.prefixlen <= p), key=lambda n: int(n.network_address))
        if not candidates:
            raise ValueError(f"no space left for {name}")
        block_ = candidates[0]
        chosen = next(block_.subnets(new_prefix=p))          # first subnet of that size
        free.remove(block_)
        free.extend(block_.address_exclude(chosen))          # give the rest back
        plan.append((name, hosts, chosen))
    return plan, sorted(free, key=lambda n: int(n.network_address))


if __name__ == "__main__":
    print("1) 192.168.50.0/24 split into 6 equal subnets (needs 3 borrowed bits -> 8 subnets)")
    for s in fixed_subnets("192.168.50.0/24", 6):
        h = list(s.hosts())
        print(f"   {str(s):18} hosts {h[0]} - {h[-1]}  broadcast {s.broadcast_address}  ({len(h)} usable)")

    print("\n2) VLSM plan for 172.16.0.0/22")
    needs = [("Lecturers", 200), ("Students lab", 400), ("Admin office", 50),
             ("Printers", 10), ("R1-R2 link", 2), ("R1-R3 link", 2)]
    plan, free = vlsm("172.16.0.0/22", needs)
    print(f"   {'Segment':14} {'Need':>5}  {'Subnet':18} {'Mask':16} {'Usable':>6}  Gateway")
    for name, hosts, net in plan:
        print(f"   {name:14} {hosts:>5}  {str(net):18} {str(net.netmask):16} {net.num_addresses - 2:>6}  {net.network_address + 1}")
    print("   Unused blocks:", ", ".join(map(str, free)))

    print("\n3) CIDR summarisation (what a router can advertise as one route)")
    routes = [ipaddress.ip_network(f"10.20.{i}.0/24") for i in range(8, 16)]
    print("   Branch routes:", ", ".join(str(r) for r in routes[:3]), "...", routes[-1])
    print("   Summary      :", ", ".join(str(n) for n in ipaddress.collapse_addresses(routes)))
    odd = [ipaddress.ip_network(n) for n in ("10.20.8.0/24", "10.20.9.0/24", "10.20.11.0/24")]
    print("   Non-contiguous", [str(n) for n in odd], "->", [str(n) for n in ipaddress.collapse_addresses(odd)])
    print("   Classful view of 172.16.0.0/22: a class B /16 would waste",
          65_534 - 1_022, "addresses compared with CIDR /22")

1) 192.168.50.0/24 split into 6 equal subnets (needs 3 borrowed bits -> 8 subnets)
   192.168.50.0/27    hosts 192.168.50.1 - 192.168.50.30  broadcast 192.168.50.31  (30 usable)
   192.168.50.32/27   hosts 192.168.50.33 - 192.168.50.62  broadcast 192.168.50.63  (30 usable)
   192.168.50.64/27   hosts 192.168.50.65 - 192.168.50.94  broadcast 192.168.50.95  (30 usable)
   192.168.50.96/27   hosts 192.168.50.97 - 192.168.50.126  broadcast 192.168.50.127  (30 usable)
   192.168.50.128/27  hosts 192.168.50.129 - 192.168.50.158  broadcast 192.168.50.159  (30 usable)
   192.168.50.160/27  hosts 192.168.50.161 - 192.168.50.190  broadcast 192.168.50.191  (30 usable)
   192.168.50.192/27  hosts 192.168.50.193 - 192.168.50.222  broadcast 192.168.50.223  (30 usable)
   192.168.50.224/27  hosts 192.168.50.225 - 192.168.50.254  broadcast 192.168.50.255  (30 usable)

2) VLSM plan for 172.16.0.0/22
   Segment         Need  Subnet             Mask             Usable  Gateway
   Students lab     400  17